In [1]:
import numpy as np
from sklearn.metrics import accuracy_score, r2_score
from sklearn.base import clone

In [ ]:
selected_accuracy = input('select accuracy or r2')

In [ ]:
selected_direction = input('select forward or backward')

In [ ]:
class SequentialFeatureSelector:

    def __init__(self, estimator, n_features_to_select, direction=selected_direction, scoring=selected_accuracy):
        self.estimator = estimator
        self.n_features_to_select = n_features_to_select
        self.direction = direction
        self.scoring = scoring
        self.selected_features_ = None

    
    def fit(self, X, y):
        self.n_features = X.shape[1]

        if self.scoring == "accuracy":
            score_func = accuracy_score
        elif self.scoring == "r2":
            score_func = r2_score
        else:
            raise ValueError("Unsupported scoring method")

        if self.direction == "forward":
            selected = []
            while len(selected) < self.n_features_to_select:
                best_score = -np.inf
                best_feature = None
                for feature in range(self.n_features):
                    if feature in selected:
                        continue
                    current_features = selected + [feature]
                    X_subset = X[:, current_features]
                    model = clone(self.estimator)
                    model.fit(X_subset, y)
                    y_pred = model.predict(X_subset)
                    score = score_func(y, y_pred)
                    if score > best_score:
                        best_score = score
                        best_feature = feature
                selected.append(best_feature)
            self.selected_features_ = selected
            
        elif self.direction == "backward":
            selected = list(range(self.n_features))
            while len(selected) > self.n_features_to_select:
                best_score = -np.inf
                worst_feature = None
                for feature in selected:
                    current_features = [f for f in selected if f != feature]
                    X_subset = X[:, current_features]
                    model = clone(self.estimator)
                    model.fit(X_subset, y)
                    y_pred = model.predict(X_subset)
                    score = score_func(y, y_pred)
                    if score > best_score:
                        best_score = score
                        worst_feature = feature
                selected.remove(worst_feature)
            self.selected_features_ = selected
            
        else:
            raise ValueError("direction must be 'forward' or 'backward'")

        return self

    
    def transform(self, X):
        return X[:, self.selected_features_]

    
    def fit_transform(self, X, y):
        self.fit(X, y)
        return self.transform(X)

    
    def get_support(self):
        mask = np.zeros(self.n_features, dtype=bool)
        mask[self.selected_features_] = True
        return mask